In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from src.config import (
    load_config,
    get_embedding_model,
    get_top_k,
)

from src.embeddings import EmbeddingService
from src.faiss_index import FaissIndexService
from src.retriever import RetrieverService

In [5]:
config = load_config()

embedding_model = get_embedding_model(config)

top_k = get_top_k(config)

In [6]:
embedding_service = EmbeddingService(
    embedding_model
)

faiss_service = FaissIndexService()

faiss_service.load(
    PROJECT_ROOT
    / "outputs"
    / "faiss"
    / "knowledge_base.faiss"
)

retriever = RetrieverService(
    embedding_service,
    faiss_service,
)

2026-07-12 11:41:45,419 | INFO | src.embeddings | Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-12 11:41:51,394 | INFO | src.embeddings | Embedding model loaded.
2026-07-12 11:41:51,396 | INFO | src.faiss_index | FAISS index loaded successfully.


In [7]:
query = "What is Machine Learning?"

distances, indices = retriever.retrieve(
    query,
    top_k,
)

2026-07-12 11:42:11,137 | INFO | src.retriever | Embedding query.
2026-07-12 11:42:11,140 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-12 11:42:12,170 | INFO | src.embeddings | Embedding generation completed.
2026-07-12 11:42:12,170 | INFO | src.retriever | Searching FAISS index.
2026-07-12 11:42:12,175 | INFO | src.retriever | Retrieved 10 documents.


In [8]:
print("Distances:")
print(distances)

print()

print("Indices:")
print(indices)

Distances:
[[1.465842  1.465842  1.465842  1.465842  1.465842  1.465842  1.465842
  1.4754894 1.4754894 1.4754894]]

Indices:
[[ 225  627  725 1096 1297 1406 1996   70  344  412]]


In [9]:
assert distances.shape == (1, top_k)
assert indices.shape == (1, top_k)